# Basic SQL ETL And Schema Walkthrough

This notebook shows the smallest useful pattern for pulling data from any SQL Server database, transforming it in Python, and writing the result back to a table in a target schema. It also explains the PricingLab schema shape and points at the current SQL Server DDL that can be imported into an ERD generator.

The important idea is that model code should not know the authentication mechanism. It asks for a SQLAlchemy engine, runs SQL, and writes rows. The shared engine helper can use local SQL auth, work SQL auth, or Entra token auth behind the same call.

## DDL And ERD Files

Use `tutorials/schema/pricing_useful_tables_ddl.sql` when an ERD tool wants plain SQL Server `CREATE TABLE` syntax. This file is a tested copy of `docs/pricing_useful_tables_ddl.sql`, so it stays aligned with the repository reference.

Use `docs/pricing_useful_tables_full_ddl.sql` when you want the fuller SQL Server contract with schemas, constraints, filtered indexes, and runtime views. That file also includes `pricing.V_CURRENT_RATE_PACKAGE`, `pricing_runtime.V_COMPILED_RATE_CELL`, `pricing_runtime.V_COMPILED_RATE_CELL_LEVEL`, and `pricing_runtime.V_COMPILED_1D_RATE_BAND`.

`pricing.PREDICT_CURRENT_RATE` lives in `db/migrations/V014__current_rate_prediction_proc.sql`; it is a stored procedure rather than a table, so most ERD tools will not draw it.

## Logical Data Flow

```mermaid
flowchart LR
    source[(Source SQL Server)] --> raw[raw.FREMTPL_RAW or model source table]
    raw --> manifest[mlops.DATASET_MANIFEST]
    manifest --> columns[mlops.DATASET_COLUMN]
    manifest --> splitset[mlops.CV_SPLIT_SET]
    splitset --> folds[mlops.CV_FOLD]
    splitset --> splitrows[mlops.CV_SPLIT_ROW]
    manifest --> run[mlops.MODEL_RUN]
    run --> rundataset[mlops.MODEL_RUN_DATASET]
    run --> runsplit[mlops.MODEL_RUN_SPLIT_SET]
    run --> metrics[mlops.MODEL_RUN_METRIC]
    run --> package[pricing.RATE_PACKAGE]
    model[pricing.MODEL] --> run
    model --> package
    package --> term[pricing.TERM]
    term --> cell[pricing.RATE_CELL]
    feature[pricing.FEATURE] --> levelset[pricing.FEATURE_LEVEL_SET]
    levelset --> level[pricing.FEATURE_LEVEL]
    cell --> celllevel[pricing.RATE_CELL_LEVEL]
    package --> deployment[pricing.MODEL_DEPLOYMENT]
    deployment --> runtime[pricing_runtime compiled views]
```


## Schema Meaning

| Schema | Main objects | Purpose |
| --- | --- | --- |
| `raw` | `raw.FREMTPL_RAW` | Raw or lightly landed source data. In work use, this may be any source table or an existing corporate database table rather than freMTPL. |
| `mlops` | `mlops.DATASET_MANIFEST`, `mlops.DATASET_COLUMN`, `mlops.CV_SPLIT_SET`, `mlops.CV_FOLD`, `mlops.CV_SPLIT_ROW`, `mlops.MODEL_RUN`, `mlops.MODEL_RUN_DATASET`, `mlops.MODEL_RUN_SPLIT_SET`, `mlops.MODEL_RUN_METRIC`, `mlops.CV_FOLD_METRIC` | Audit layer. It answers what data was used, what split definition was used, which rows were in each test fold when materialized, and which model run consumed those assets. |
| `pricing` | `pricing.MODEL`, `pricing.RATE_PACKAGE`, `pricing.FEATURE`, `pricing.FEATURE_LEVEL_SET`, `pricing.FEATURE_LEVEL`, `pricing.TERM`, `pricing.TERM_FEATURE`, `pricing.RATE_CELL`, `pricing.RATE_CELL_LEVEL`, `pricing.MODEL_DEPLOYMENT` | Persisted pricing model and rating table structure. This is the historical record of all models, packages, terms, features, cells, and deployments. |
| `pricing_runtime` | `pricing_runtime.V_COMPILED_RATE_CELL`, `pricing_runtime.V_COMPILED_RATE_CELL_LEVEL`, `pricing_runtime.V_COMPILED_1D_RATE_BAND` | Read-optimized views for inspection, downstream rating lookups, and SQL prediction validation. |

`pricing.MODEL` is the model family. `mlops.MODEL_RUN` is one training attempt. `pricing.RATE_PACKAGE` is the published rating table output from a model run or manual adjustment. `pricing.MODEL_DEPLOYMENT` says which package is current for a slot such as UAT or PROD.

## Basic ETL Shape

The ETL should be direct: read SQL, transform a dataframe, write SQL. Keep the model-specific query and transformation in that model's folder, and keep connection mechanics in `pricing_pipeline/infra/db.py`.

In [1]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pricing_pipeline").is_dir() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pricing_pipeline/")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
REPO_ROOT


PosixPath('/home/mhick/python_projects/airflow_superglm_builder')

In [2]:
from __future__ import annotations

import os
import re

import numpy as np
import pandas as pd
from sqlalchemy import text

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

from pricing_pipeline.infra.config import Settings
from pricing_pipeline.infra.db import get_engine


## Show The Schema From The DDL

This parses the ERD-friendly DDL into visible notebook tables. The point is to see the persisted model shape without relying on a separate ERD image.

In [3]:
DDL_PATH = Path("tutorials/schema/pricing_useful_tables_ddl.sql")


def parse_ddl_schema(ddl_text: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    table_pattern = re.compile(
        r"CREATE TABLE (?P<table>[a-z_]+\.[A-Z0-9_]+) \(\n(?P<body>.*?)\n\);",
        re.S,
    )
    fk_pattern = re.compile(
        r"FOREIGN KEY \((?P<source_columns>[^)]*)\) "
        r"REFERENCES (?P<target_table>[a-z_]+\.[A-Z0-9_]+)"
        r"\((?P<target_columns>[^)]*)\)",
        re.S,
    )
    skipped = {"PRIMARY", "FOREIGN", "UNIQUE", "CONSTRAINT", "CHECK"}
    table_rows = []
    column_rows = []
    fk_rows = []

    for table_match in table_pattern.finditer(ddl_text):
        table_name = table_match.group("table")
        schema_name, short_name = table_name.split(".", 1)
        body = table_match.group("body")
        table_rows.append({"schema": schema_name, "table": short_name, "full_name": table_name})

        ordinal_no = 0
        for raw_line in body.splitlines():
            line = raw_line.strip().rstrip(",")
            if not line:
                continue
            first_token = line.split()[0].upper()
            if first_token in skipped:
                continue
            ordinal_no += 1
            parts = line.split(None, 2)
            column_rows.append(
                {
                    "table": table_name,
                    "ordinal_no": ordinal_no,
                    "column_name": parts[0],
                    "sql_type": parts[1] if len(parts) > 1 else "",
                    "definition": line,
                }
            )

        for fk_match in fk_pattern.finditer(body):
            fk_rows.append(
                {
                    "source_table": table_name,
                    "source_columns": " + ".join(
                        col.strip() for col in fk_match.group("source_columns").split(",")
                    ),
                    "target_table": fk_match.group("target_table"),
                    "target_columns": " + ".join(
                        col.strip() for col in fk_match.group("target_columns").split(",")
                    ),
                }
            )

    return pd.DataFrame(table_rows), pd.DataFrame(column_rows), pd.DataFrame(fk_rows)


schema_tables, schema_columns, schema_foreign_keys = parse_ddl_schema(
    DDL_PATH.read_text(encoding="utf-8")
)
display(schema_tables)
display(schema_columns.head(30))
display(schema_foreign_keys)


,schema,table,full_name
0,raw,FREMTPL_RAW,raw.FREMTPL_RAW
1,mlops,DATASET_MANIFEST,mlops.DATASET_MANIFEST
2,mlops,DATASET_COLUMN,mlops.DATASET_COLUMN
3,pricing,MODEL,pricing.MODEL
4,mlops,MODEL_RUN,mlops.MODEL_RUN
5,mlops,CV_SPLIT_SET,mlops.CV_SPLIT_SET
6,mlops,CV_FOLD,mlops.CV_FOLD
7,mlops,CV_SPLIT_ROW,mlops.CV_SPLIT_ROW
8,mlops,MODEL_RUN_DATASET,mlops.MODEL_RUN_DATASET
9,mlops,MODEL_RUN_SPLIT_SET,mlops.MODEL_RUN_SPLIT_SET


,table,ordinal_no,column_name,sql_type,definition
0,raw.FREMTPL_RAW,1,IDpol,BIGINT,IDpol BIGINT PRIMARY KEY
1,raw.FREMTPL_RAW,2,ClaimNb,INT,ClaimNb INT NOT NULL
2,raw.FREMTPL_RAW,3,Exposure,FLOAT,Exposure FLOAT NOT NULL
3,raw.FREMTPL_RAW,4,Area,NVARCHAR(16),Area NVARCHAR(16)
4,raw.FREMTPL_RAW,5,VehPower,INT,VehPower INT
5,raw.FREMTPL_RAW,6,VehAge,INT,VehAge INT
6,raw.FREMTPL_RAW,7,DrivAge,INT,DrivAge INT
7,raw.FREMTPL_RAW,8,BonusMalus,INT,BonusMalus INT
8,raw.FREMTPL_RAW,9,VehBrand,NVARCHAR(64),VehBrand NVARCHAR(64)
9,raw.FREMTPL_RAW,10,VehGas,NVARCHAR(16),VehGas NVARCHAR(16)


,source_table,source_columns,target_table,target_columns
0,mlops.DATASET_COLUMN,manifest_id,mlops.DATASET_MANIFEST,manifest_id
1,mlops.MODEL_RUN,model_id,pricing.MODEL,model_id
2,mlops.CV_SPLIT_SET,manifest_id,mlops.DATASET_MANIFEST,manifest_id
3,mlops.CV_FOLD,split_set_id,mlops.CV_SPLIT_SET,split_set_id
4,mlops.CV_SPLIT_ROW,split_set_id,mlops.CV_SPLIT_SET,split_set_id
5,mlops.CV_SPLIT_ROW,split_set_id + test_fold_no,mlops.CV_FOLD,split_set_id + fold_no
6,mlops.MODEL_RUN_DATASET,model_run_id,mlops.MODEL_RUN,model_run_id
7,mlops.MODEL_RUN_DATASET,manifest_id,mlops.DATASET_MANIFEST,manifest_id
8,mlops.MODEL_RUN_SPLIT_SET,model_run_id,mlops.MODEL_RUN,model_run_id
9,mlops.MODEL_RUN_SPLIT_SET,model_run_id + dataset_role + manifest_id,mlops.MODEL_RUN_DATASET,model_run_id + dataset_role + manifest_id


In [4]:
SOURCE_SQL = """
SELECT
    policy_id,
    accounting_month,
    exposure,
    claim_count,
    vehicle_age,
    driver_age,
    rating_area
FROM source_schema.PolicyClaims
WHERE accounting_month >= :from_month
  AND exposure > 0
"""


def transform_source_rows(raw: pd.DataFrame) -> pd.DataFrame:
    frame = raw.copy()
    frame["log_exposure"] = np.log(frame["exposure"].astype(float).clip(lower=1e-12))
    frame["vehicle_age_band"] = pd.cut(
        frame["vehicle_age"],
        bins=[0, 1, 3, 7, 15, 100],
        right=False,
    ).astype(str)
    return frame


## Offline End-To-End Demo

These cells make the workflow visible without touching a real SQL Server. Treat `source_result_set` as the rows returned by `pd.read_sql_query`, `transformed_result_set` as the Python ETL output, `rating_output` as the model/rating step, and `load_preview` as the rows that would be written back to SQL Server.

In [5]:
source_result_set = pd.DataFrame(
    [
        {
            "policy_id": 1001,
            "accounting_month": "2025-01",
            "exposure": 0.75,
            "claim_count": 0,
            "vehicle_age": 2,
            "driver_age": 41,
            "rating_area": "A",
        },
        {
            "policy_id": 1002,
            "accounting_month": "2025-01",
            "exposure": 1.00,
            "claim_count": 1,
            "vehicle_age": 11,
            "driver_age": 23,
            "rating_area": "C",
        },
        {
            "policy_id": 1003,
            "accounting_month": "2025-02",
            "exposure": 0.50,
            "claim_count": 0,
            "vehicle_age": 6,
            "driver_age": 68,
            "rating_area": "B",
        },
    ]
)
source_result_set


,policy_id,accounting_month,exposure,claim_count,vehicle_age,driver_age,rating_area
0,1001,2025-01,0.75,0,2,41,A
1,1002,2025-01,1.00,1,11,23,C
2,1003,2025-02,0.50,0,6,68,B


In [6]:
transformed_result_set = transform_source_rows(source_result_set)
transformed_result_set


,policy_id,accounting_month,exposure,claim_count,vehicle_age,driver_age,rating_area,log_exposure,vehicle_age_band
0,1001,2025-01,0.75,0,2,41,A,-0.287682,"[1, 3)"
1,1002,2025-01,1.00,1,11,23,C,0.000000,"[7, 15)"
2,1003,2025-02,0.50,0,6,68,B,-0.693147,"[3, 7)"


In [7]:
area_factor = {"A": 0.95, "B": 1.05, "C": 1.20}
rating_output = transformed_result_set.assign(
    MODEL_SCORE=lambda frame: frame["exposure"]
    * np.exp(
        -2.1
        + 0.015 * frame["vehicle_age"].astype(float)
        + 0.004 * frame["driver_age"].astype(float)
        + np.log(frame["rating_area"].map(area_factor).fillna(1.0))
    )
)
rating_output.loc[
    :,
    [
        "policy_id",
        "exposure",
        "vehicle_age_band",
        "rating_area",
        "MODEL_SCORE",
    ],
]


,policy_id,exposure,vehicle_age_band,rating_area,MODEL_SCORE
0,1001,0.75,"[1, 3)",A,0.105930
1,1002,1.00,"[7, 15)",C,0.190010
2,1003,0.50,"[3, 7)",B,0.092333


In [8]:
TARGET_SCHEMA = "mlops"
TARGET_TABLE = "MY_MODEL_SCORING_OUTPUT"

load_preview = rating_output.loc[
    :,
    [
        "policy_id",
        "accounting_month",
        "exposure",
        "claim_count",
        "vehicle_age_band",
        "MODEL_SCORE",
    ],
].copy()
load_preview["model_key"] = "MY_MODEL"
load_preview["model_version"] = "demo_20260512"

print(f"Would append {len(load_preview)} rows to {TARGET_SCHEMA}.{TARGET_TABLE}")
load_preview


Would append 3 rows to mlops.MY_MODEL_SCORING_OUTPUT


,policy_id,accounting_month,exposure,claim_count,vehicle_age_band,MODEL_SCORE,model_key,model_version
0,1001,2025-01,0.75,0,"[1, 3)",0.105930,MY_MODEL,demo_20260512
1,1002,2025-01,1.00,1,"[7, 15)",0.190010,MY_MODEL,demo_20260512
2,1003,2025-02,0.50,0,"[3, 7)",0.092333,MY_MODEL,demo_20260512


## Read From One SQL Server And Write To Another

`source_database` and `target_database` can be the same database or different databases on the same SQL Server. If the target is a different SQL Server instance, create a second settings object or extend the shared helper so it accepts a server override. The ETL code still only deals with engines.

In [9]:
def run_basic_sql_etl(
    *,
    from_month: str,
    source_database: str,
    target_database: str,
    target_schema: str,
    target_table: str,
) -> int:
    settings = Settings.from_env(os.environ)

    source_engine = get_engine(settings, database=source_database)
    target_engine = get_engine(settings, database=target_database)

    raw = pd.read_sql_query(
        text(SOURCE_SQL),
        source_engine,
        params={"from_month": from_month},
    )
    transformed = transform_source_rows(raw)

    with target_engine.begin() as con:
        transformed.to_sql(
            target_table,
            con,
            schema=target_schema,
            if_exists="append",
            index=False,
            chunksize=10_000,
        )

    return len(transformed)


## Chunked Version For Larger Tables

For large tables, stream chunks from SQL Server and write each transformed chunk. This avoids holding the entire source dataset in memory.

In [10]:
def run_chunked_sql_etl(
    *,
    from_month: str,
    source_database: str,
    target_database: str,
    target_schema: str,
    target_table: str,
    read_chunksize: int = 100_000,
) -> int:
    settings = Settings.from_env(os.environ)
    source_engine = get_engine(settings, database=source_database)
    target_engine = get_engine(settings, database=target_database)
    rows_written = 0

    with target_engine.begin() as con:
        for raw_chunk in pd.read_sql_query(
            text(SOURCE_SQL),
            source_engine,
            params={"from_month": from_month},
            chunksize=read_chunksize,
        ):
            transformed = transform_source_rows(raw_chunk)
            transformed.to_sql(
                target_table,
                con,
                schema=target_schema,
                if_exists="append",
                index=False,
                chunksize=10_000,
            )
            rows_written += len(transformed)

    return rows_written


## Where This Lives For A Real Model

For a model called `my_model`, keep the model-specific query and transform in `pricing_models/my_model/etl.py`. The DAG should call a small function from that file. Shared connection logic stays in `pricing_pipeline/infra/db.py`; shared audit writers stay under `pricing_pipeline/`.

A real DAG task can call `run_basic_sql_etl(...)` before the model training task. The model training task then reads the prepared table using its own `training_sql`.

In [11]:
# Example DAG task body, not executed in this notebook.
def airflow_task_build_training_table() -> int:
    return run_basic_sql_etl(
        from_month="2025-01",
        source_database=os.environ["SOURCE_DATABASE"],
        target_database=os.environ["TARGET_DATABASE"],
        target_schema="mlops",
        target_table="MY_MODEL_TRAINING_DATA",
    )


## Work Authentication Note

If work SQL Server access uses Entra token auth, do not put that in every ETL task. Extend `pricing_pipeline/infra/db.py` so `get_engine(...)` can create an engine with `connect_args={"attrs_before": {1256: token_struct}}`. The constant `1256` is the Microsoft ODBC driver attribute id for an access token; it is not a secret.

Once the helper supports that mode, ETL code remains unchanged:

```python
settings = Settings.from_env(os.environ)
engine = get_engine(settings, database="ExistingWorkDatabase")
```


## Practical Rules

- Use `append` for audit/history tables unless you intentionally rebuild a scratch table.
- Use explicit SQL column lists instead of `SELECT *` for work pipelines.
- Keep source SQL and transform code close to the model that owns it.
- Keep all model runs historical; expose current state through views such as `pricing.V_CURRENT_RATE_PACKAGE`.
- Validate SQL prediction output against `SuperGLM.predict(X, offset=np.log(exposure))` with `scripts/validate_sql_prediction_against_superglm.py` before trusting a deployed rating package.